In [ ]:
# 사전 설치 : pip install Faker tabulate
# Faker 데이터 활용하는 경우 : 실제 데이터 확보가 어려운 경우(개인정보, 초기에 서비스가 없어서 로그 및 사용자 데이터가 없는 경우)
# Faker 데이터는 모델 성능 최적화용이라기보다 데이터 파이프라인 정상 동작 여부 및 학습 코드 예측 오류 검증 용도로 활용
# Faker + 실제 데이터 혼합 전략 : 초기(100% Faker,모델 구조, API, 파이프라인 검증), 중간(Faker 70% + 실제 30%, 데이터 부족 구간 보완), 운영(실제 80~95%, Faker는 희귀케이스만 유지)
import pandas as pd
from faker import Faker
import random
import os

# 1. 설정
NUM_SAMPLES = 1000  # 생성할 데이터 개수
OUTPUT_FILENAME = 'fake_emr_data.csv'
LOCALE = 'ko_KR' # 한국어 로케일 설정

# 2. Faker 인스턴스 초기화
fake = Faker(LOCALE)

def generate_emr_record(fake):
    """하나의 가상 EMR(전자의무기록) 레코드를 생성하는 함수"""

    # 임의의 검사 결과값 생성
    wbc_result = round(random.uniform(3.5, 12.0), 2)
    hba1c_result = round(random.uniform(4.5, 15.0), 1)

    record = {
        'Patient_ID': f"PID-{fake.unique.random_number(digits=6)}",
        'Name': fake.name(),
        'Gender': random.choice(['Male', 'Female']),
        'Birthdate': fake.date_of_birth(minimum_age=1, maximum_age=90).strftime('%Y-%m-%d'),
        'Address': fake.address(),
        'Visit_Date': fake.date_this_year().strftime('%Y-%m-%d'),
        'Chief_Complaint': random.choice([
            '열과 오한', '지속적인 기침', '심한 두통', '복통 및 소화불량',
            '무릎 통증', '만성 피로', '고혈압 검진', '당뇨병 관리'
        ]),
        'Diagnosis': random.choice([
            '감기(Common Cold)', '급성 기관지염', '편두통', '위염',
            '골관절염', '불안 장애', '고혈압', '2형 당뇨병'
        ]),
        'Lab_Result_WBC': wbc_result, # 백혈구 수치 (4.0 - 11.0이 정상 범위)
        'Lab_Result_HbA1c': hba1c_result, # 당화혈색소 수치 (5.7 미만이 정상)
        'Medication': random.choice([
            '항생제(Amoxicillin)', '해열제(Acetaminophen)', '혈압약(Losartan)',
            '당뇨약(Metformin)', '진통제(Ibuprofen)', '처방 없음'
        ]),
        'Doctor_Note': fake.text(max_nb_chars=300) # 의사 소견 (자유 형식 텍스트)
    }

    # 임의의 소견 추가
    if wbc_result > 11.0:
        record['Doctor_Note'] += ' - **WBC 수치 상승** 확인. 감염 의심.'
    if hba1c_result >= 6.5:
         record['Doctor_Note'] += ' - **HbA1c 고수치** 확인. 당뇨병 관리 필요.'

    return record

# 3. 데이터 생성
print(f"{NUM_SAMPLES}개의 가상 EMR 데이터를 생성 중...")
emr_data = [generate_emr_record(fake) for _ in range(NUM_SAMPLES)]

# 4. DataFrame으로 변환
emr_df = pd.DataFrame(emr_data)

# 5. CSV 파일로 저장
try:
    emr_df.to_csv(OUTPUT_FILENAME, index=False, encoding='utf-8-sig')
    print("-" * 40)
    print(f"✅ 데이터 생성 완료!")
    print(f"파일 이름: {OUTPUT_FILENAME}")
    print(f"저장 경로: {os.path.abspath(OUTPUT_FILENAME)}")
    print(f"첫 5개 행 데이터 미리보기:\n")
    print(emr_df.head().to_markdown(index=False))

except Exception as e:
    print(f"❌ 파일 저장 중 오류 발생: {e}")

1000개의 가상 EMR 데이터를 생성 중...
----------------------------------------
✅ 데이터 생성 완료!
파일 이름: fake_emr_data.csv
저장 경로: c:\aisource\fake_emr_data.csv
첫 5개 행 데이터 미리보기:

| Patient_ID   | Name   | Gender   | Birthdate   | Address                                   | Visit_Date   | Chief_Complaint   | Diagnosis   |   Lab_Result_WBC |   Lab_Result_HbA1c | Medication            | Doctor_Note                                                                                                                                                                 |
|:-------------|:-------|:---------|:------------|:------------------------------------------|:-------------|:------------------|:------------|-----------------:|-------------------:|:----------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| PID-439332   | 김서준 | Female   | 1973-11-07  | 경상북도 구리시 삼성길 323 (예원김마을)   